<a href="https://colab.research.google.com/github/garykbrixi/minerva/blob/main/examples/notebooks/loci_viewer_hf.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Minerva loci viewer (HF-backed)

This notebook loads Minerva from the private Hugging Face repo `gbrixi/minerva-1` at a pinned revision. Google Drive only needs this notebook; model code, viewer utilities, and example GenBank files come from the HF snapshot.

Grant users read access to the HF repo and have them add a Colab Secret named `HF_TOKEN`.

In [ ]:
#@title Setup — install, authenticate, and load HF snapshot { display-mode: "form" }
import os, sys, gc, inspect, importlib.util, shutil
import torch

IN_COLAB = importlib.util.find_spec("google.colab") is not None
REPO_ID = "gbrixi/minerva-1"  #@param {type:"string"}
REVISION = "main"  #@param {type:"string"}
install_flash_attn = False  #@param {type:"boolean"}
#@markdown Leave `install_flash_attn` off for the most robust Colab/T4 path; this uses torch SDPA.

if IN_COLAB:
    !pip -q install "transformers>=4.41" "huggingface_hub>=0.23" safetensors biopython plotly
    if install_flash_attn and importlib.util.find_spec("flash_attn") is None:
        !pip -q install flash-attn --no-build-isolation

from huggingface_hub import get_token, login, snapshot_download
from transformers import AutoTokenizer, AutoModelForMaskedLM

HF_TOKEN = None
try:
    if IN_COLAB:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or get_token()
if HF_TOKEN:
    login(token=HF_TOKEN, new_session=False)
else:
    login(new_session=False)
    HF_TOKEN = get_token()
if not HF_TOKEN:
    raise RuntimeError("No Hugging Face token found after login.")

# Avoid stale in-memory remote-code modules when rerunning cells.
for _name in list(sys.modules):
    if _name.startswith("transformers_modules.gbrixi.minerva_hyphen_1"):
        sys.modules.pop(_name, None)
for _old in ("model", "tokenizer", "sequence", "all_tokens"):
    globals().pop(_old, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

SNAPSHOT_DIR = snapshot_download(repo_id=REPO_ID, revision=REVISION, token=HF_TOKEN)
if SNAPSHOT_DIR not in sys.path:
    sys.path.insert(0, SNAPSHOT_DIR)

from data import extract_and_tokenize_gb
from visualization import (
    head_contacts_rgb, render_fingerprints,
    publication_head_contacts_rgb,
    plot_locus, plot_publication_locus, interactive_overlay,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = {"fp16": torch.float16, "bf16": torch.bfloat16, "fp32": torch.float32}['fp16']

tokenizer = AutoTokenizer.from_pretrained(REPO_ID, revision=REVISION, token=HF_TOKEN)
model = AutoModelForMaskedLM.from_pretrained(
    REPO_ID,
    revision=REVISION,
    code_revision=REVISION,
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=DTYPE,
).to(DEVICE).eval()

_model_mod = inspect.getmodule(type(model))
_code_path = inspect.getfile(_model_mod)
if not install_flash_attn:
    _model_mod._HAS_FLASH = False
    _model_mod._HAS_FLASH_ROTARY = False

if REVISION not in (None, "main"):
    assert REVISION in _code_path, f"Stale HF remote code loaded: {_code_path}"
assert hasattr(model, "get_fingerprints"), "Loaded Minerva code lacks get_fingerprints; update REVISION or the HF repo."
assert callable(render_fingerprints), "Loaded visualization code lacks render_fingerprints; update REVISION or the HF repo."
assert 'hidden_states.float()' in inspect.getsource(_model_mod.rmsnorm_func)
assert 'not _HAS_FLASH or x.device.type != "cuda"' in inspect.getsource(_model_mod.Attention.forward)

vocab = tokenizer.get_vocab()
nuc_ids = [vocab[c] for c in "atgc" if c in vocab]
aa_chars = list("ACDEFGHIKLMNPQRSTVWY")
aa_ids = [vocab[c] for c in aa_chars if c in vocab]

print("snapshot:", SNAPSHOT_DIR)
print("code:", _code_path)
print("device/dtype:", next(model.parameters()).device, next(model.parameters()).dtype)
print("flash-attn enabled:", _model_mod._HAS_FLASH, "rotary:", _model_mod._HAS_FLASH_ROTARY)
print("heads:", sorted(model.linear_heads.keys()))


In [ ]:
#@title 1. Choose locus and view { display-mode: "form", run: "auto" }
example  = "ug27"           #@param ["ug27", "twoayggay", "upload your own"]
view     = "heads (fast)"   #@param ["heads (fast)", "jacobian (detailed)", "both"]
head_set = "l2 (last-2)"    #@param ["l2 (last-2)", "l6 (last-6)"]
renderer = "publication (PDF)"   #@param ["publication (PDF)", "static (PDF)", "interactive (zoom/pan)"]
contrast = "raw probabilities"   #@param ["raw probabilities", "auto contrast"]
#@markdown Window in token positions. Leave both `0` for the preset/full locus.
record       = 0  #@param {type:"integer"}
window_start = 0  #@param {type:"integer"}
window_end   = 0  #@param {type:"integer"}
jac_max_tokens = 384  #@param {type:"integer"}
dpi = 600             #@param {type:"integer"}


In [ ]:
#@title Optional upload your own GenBank { display-mode: "form" }
UPLOADED = None
if IN_COLAB and example == "upload your own":
    from google.colab import files
    up = files.upload()
    if up:
        UPLOADED = list(up.keys())[0]
        print("uploaded:", UPLOADED)


In [ ]:
#@title 2. Load selected locus { display-mode: "form" }
def _snapshot_example(name):
    p = os.path.join(SNAPSHOT_DIR, "examples", name)
    if not os.path.exists(p):
        raise FileNotFoundError(p)
    return p

PRESETS = {
    "ug27": dict(gb=_snapshot_example("UG27_systems.gb"), record=2, window=(748, 1772)),
    "twoayggay": dict(gb=_snapshot_example("TwoAYGGAY_Pseudomonas_fluorescens_SBW25.gb"), record=0, window=None),
}
if example == "upload your own":
    assert UPLOADED, "Run the upload cell first, or choose a built-in example."
    cfg = dict(gb=UPLOADED, record=record, window=None)
else:
    cfg = dict(PRESETS[example])
if window_end > window_start:
    cfg["window"] = (window_start, window_end)

records = extract_and_tokenize_gb(cfg["gb"], use_existing_translations=True)
record_obj = records[cfg["record"]]
sequence = record_obj["sequence"]
all_tokens = tokenizer.convert_ids_to_tokens(tokenizer.encode(sequence))
ntok = len(all_tokens)
window = cfg["window"]
suffix = "_l6" if head_set.startswith("l6") else ""
locus = record_obj["locus_name"][:44]
print("file:", cfg["gb"])
print("locus:", locus)
print("tokens:", ntok, "| window:", window or "(whole)", "| heads:", head_set, "| render:", renderer)

def render(rgb, tokens, title, tag, overlay_kind="heads"):
    if renderer.startswith("interactive"):
        interactive_overlay(rgb, title=title).show()
    elif renderer.startswith("publication"):
        plot_publication_locus(rgb, title=title, overlay_kind=overlay_kind, save=f"{tag}.pdf", dpi=dpi)
        import matplotlib.pyplot as plt
        plt.show(); print("saved", f"{tag}.pdf")
    else:
        plot_locus(rgb, tokens=tokens, title=title, save=f"{tag}.pdf", dpi=dpi)
        import matplotlib.pyplot as plt
        plt.show(); print("saved", f"{tag}.pdf")


In [ ]:
#@title 3. Heads view (fast) { display-mode: "form" }
if view.startswith("heads") or view == "both":
    import numpy as np
    heads = [f"base_pairing{suffix}", f"repeat{suffix}", f"protein{suffix}"]
    kw = dict(sequence=sequence, tokenizer=tokenizer, head_names=heads, return_dict=True)
    if window:
        kw.update(seed_start=window[0], seed_end=window[1])
    with torch.no_grad():
        pr = model.predict_contacts(**kw)["predictions"]
    chan = {h.replace(suffix, ""): pr[h].float().cpu().numpy() for h in heads}
    toks = all_tokens[window[0]:window[1]] if window else all_tokens
    print("head ranges:", {k: (float(np.nanmin(v)), float(np.nanmax(v))) for k, v in chan.items()})
    if renderer.startswith("publication"):
        use_auto_contrast = contrast.startswith("auto")
        rgb_heads, contrast_stats = publication_head_contacts_rgb(
            chan, tokens=toks, auto_contrast=use_auto_contrast, return_stats=True)
        print("contrast:", contrast_stats if use_auto_contrast else "raw probabilities")
    else:
        rgb_heads = head_contacts_rgb(chan, tokens=toks)
    nonwhite = int(np.sum(np.any(rgb_heads < 0.995, axis=-1)))
    print("visible pixels:", nonwhite, "of", rgb_heads.shape[0] * rgb_heads.shape[1])
    if nonwhite == 0:
        print("warning: selected heads produced no visible structure; try l6, auto contrast, or jacobian mode")
    render(rgb_heads, toks, f"Heads {head_set} - {locus} {window or '(whole)'}", "heads", overlay_kind="heads")


In [ ]:
#@title 4. Jacobian fingerprint view (detailed, slow) { display-mode: "form" }
if view.startswith("jacobian") or view == "both":
    import time
    if window:
        js, je = window[0], min(window[1], window[0] + jac_max_tokens)
    else:
        c = ntok // 2
        js, je = max(0, c - jac_max_tokens // 2), min(ntok, c + jac_max_tokens // 2)
    print(f"jacobian window [{js}:{je}] = {je - js} tokens")
    t0 = time.time()
    fp = model.get_fingerprints(
        sequence, tokenizer, nuc_token_ids=nuc_ids, aa_token_ids=aa_ids,
        max_batch_size=32, position_range=(js, je), show_progress=True,
        autocast_dtype=DTYPE if DEVICE == "cuda" and DTYPE != torch.float32 else None,
        jac_aa_order=aa_chars,
    )
    jac_tokens = fp.tokens
    style = "publication" if renderer.startswith("publication") else "default"
    rgb_jac = render_fingerprints(fp, style=style)
    print(f"done in {time.time() - t0:.0f}s")
    render(rgb_jac, jac_tokens, f"Jacobian fingerprint - {locus} [{js}:{je}]", "jacobian", overlay_kind="fingerprint")


In [ ]:
#@title Optional attention extraction sanity check { display-mode: "form" }
run_attention_check = False  #@param {type:"boolean"}
if run_attention_check:
    seq = "<+>atgcatgcaaaaaatttaaaaaatgcatgc"
    ids = tokenizer.encode(seq, return_tensors="pt").to(next(model.parameters()).device)
    with torch.no_grad():
        attn = model.get_attention_maps(input_ids=ids, layers=[31, 32])
    for layer, a in attn.items():
        x = a.detach().float().cpu()
        print(layer, tuple(x.shape), x.min().item(), x.max().item(), x.mean().item(), x.std().item(),
              "nan", torch.isnan(x).sum().item(), "inf", torch.isinf(x).sum().item())


In [ ]:
#@title Download figures { display-mode: "form" }
if IN_COLAB:
    from google.colab import files
    for f in ("heads.pdf", "jacobian.pdf"):
        if os.path.exists(f):
            files.download(f)


---
**Notes**

- The default renderer uses raw 0..1 head probabilities. Auto contrast is exploratory only.
- This notebook does not use Drive-local Minerva code. It imports model code, data helpers, visualization helpers, and examples from the pinned HF snapshot.
- If Colab shows an old remote-code hash, restart the runtime and rerun from the top.
- To share confidentially, grant HF read access and have users add `HF_TOKEN` as a Colab Secret.